In [1]:
from pyspark.sql import SparkSession
from delta import *
import os

In [2]:
def create_spark_session():
    """
    Creates a SparkSession with Delta Lake and S3/MinIO configurations.
    """
    builder = SparkSession.builder \
        .appName("EnterpriseLakehouseInit") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.sql.warehouse.dir", "s3a://warehouse/")

    # Add required Maven packages for Delta + AWS S3
    # Note: Versions must be compatible with the Spark version in the Dockerfile (3.5.0)
    return configure_spark_with_delta_pip(builder, extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ]).getOrCreate()

In [3]:
spark = create_spark_session()

In [4]:
def write_table(spark, table_name, df, write_mode="overwrite"):
    spark.conf.set("spark.databricks.delta.clusteredTable.enableClusteringTablePreview", "false")
    table_path = f"s3a://warehouse/{table_name}"
    print(f"Writing Delta table to {table_path}...")
    df.write.format("delta").mode(write_mode).save(table_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{table_path}'")
    print(f"Table {table_name} created successfully.")

def write_clustered_table(spark, table_name, df, write_mode="overwrite"):
    spark.conf.set("spark.databricks.delta.clusteredTable.enableClusteringTablePreview", "true")
    table_path = f"s3a://warehouse/{table_name}"
    if write_mode == "overwrite":
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name}(id long, name string, department string, hire_date string) USING DELTA CLUSTER BY (name, department) LOCATION '{table_path}'")
    print(f"Writing Delta table to {table_path}...")
    df.write.format("delta").option("clusteringColumns", "name, department").mode(write_mode).save(table_path)
    print(f"Table {table_name} created successfully.")

In [5]:
def print_table(spark, table_name):
    print("\n--- Verifying Data via SQL ---")
    df = spark.sql(f"SELECT * FROM {table_name}")
    df.show(df.count())

def print_delta_history(spark, table_name):
    print("\n--- Verifying Delta History ---")
    from delta.tables import DeltaTable
    dt = DeltaTable.forPath(spark, f"s3a://warehouse/{table_name}")
    dt.history().show()

def optimize_table(spark, table_name):
    print("\n--- Optimizing table via SQL ---")
    spark.sql(f"OPTIMIZE {table_name}")

def vacuum_files(spark, table_name):
    print("\n--- Vacuuming files up ---")
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    from delta.tables import DeltaTable
    dt = DeltaTable.forPath(spark, f"s3a://warehouse/{table_name}")
    dt.vacuum(retentionHours=0)


In [9]:
# Define table name
table_name = "test21"

# Define data schema
columns = ["id", "name", "department", "hire_date"]

# Define DF-1
data1 = [
    (1, "Alice", "Engineering", "2023-01-01"),
    (2, "Bob", "Sales", "2023-01-02"),
    (3, "Charlie", "Marketing", "2023-01-03")
]
df1 = spark.createDataFrame(data1, columns)

write_clustered_table(spark, table_name, df1)
for i in range(3,600,3):
    # Define DF-2
    data2 = [
        (i+1, "Alice", "Engineering-Team", "2023-01-01"),
        (i+2, "Charlie", "Flight", "2023-01-02"),
        (i+3, "Charlie", "Marketing", "2023-01-03")
    ]
    df2 = spark.createDataFrame(data2, columns)
    write_clustered_table(spark, table_name, df2, write_mode="append")

optimize_table(spark, table_name)
vacuum_files(spark, table_name)
print_table(spark, table_name)
print_delta_history(spark, table_name)



Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created successfully.
Writing Delta table to s3a://warehouse/test21...
Table test21 created succes